# Customer Intelligence & Support Copilot
## Instructor-led end-to-end notebook

**Scenario:** synthetic grocery quick-commerce data. This notebook emphasizes *how data is fetched, inspected, cleaned, validated, stored, and then consumed by downstream AI/ML components*.

### Before class
1. Run `python generate_data.py` from the repository root.
2. Install packages with `pip install -r requirements.txt`.
3. Run this notebook from the repository root so relative paths work.

All data and policies are fictional. The LLM calls are handled by `mock_llm.py` locally; no API key is required.

## 1. How the project works

The project is organised as a series of connected stages:

1. Sample customer, product, order and support data is created.
2. The original data is checked for errors and inconsistencies.
3. The data is cleaned and saved in CSV files and an SQLite database.
4. Customer purchasing patterns are analysed and customers are grouped into segments.
5. A model is used to estimate which customers may stop purchasing.
6. Support tickets are checked, sensitive details are masked, and tickets are directed for handling.
7. Company policy documents are searched to help answer policy-related questions.
8. A refund assistant demonstrates how an order and the relevant policy can be checked before a refund is issued or sent for human review.

The original sample files are stored in `data/raw/`. The cleaned files are stored in `data/clean/`, and the SQLite database is saved as `data/instamart.db`.

In this project, data is read from local CSV files rather than downloaded from an external service. A real business could connect a similar process to an approved API or data warehouse.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "generate_data.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Root files:", sorted(p.name for p in ROOT.iterdir() if p.is_file()))

Project root: d:\AI\Doc\Project\instamart_copilot_Project
Root files: ['.gitignore', 'Customer_Intelligence_Support_Copilot.ipynb', 'GITHUB_RESOURCES.md', 'README.md', 'generate_data.py', 'mock_llm.py', 'requirements.txt', 'run_project.py']


## 1. Generate / fetch the raw data

The data generator is deterministic (fixed random seed). It writes customer, product, order, support-ticket, evaluation-question, agent-scenario inputs and policy text files.

Run it once when starting from a clean clone. Running it again recreates the same synthetic source exports.

In [2]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / "generate_data.py")], check=True)
raw_dir = ROOT / "data" / "raw"
print("Raw files:", sorted(p.name for p in raw_dir.iterdir()))

Raw files: ['customers.csv', 'orders.csv', 'policy_eval.csv', 'products.csv', 'tickets.csv']


## 2. Read the exports and understand their shape

Use `keep_default_na=False` for ticket text so empty strings remain empty strings. This is important because empty tickets must be identified by an explicit guardrail rather than silently changing the input representation.

In [3]:
import pandas as pd
customers_raw = pd.read_csv(raw_dir / "customers.csv", keep_default_na=False)
products_raw = pd.read_csv(raw_dir / "products.csv", keep_default_na=False)
orders_raw = pd.read_csv(raw_dir / "orders.csv", keep_default_na=False)
tickets_raw = pd.read_csv(raw_dir / "tickets.csv", keep_default_na=False)

for name, df in [("customers", customers_raw), ("products", products_raw),
                 ("orders", orders_raw), ("tickets", tickets_raw)]:
    print(f"\\n{name}: shape={df.shape}")
    display(df.head(3))
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "blank_count": df.astype(str).apply(lambda col: col.str.strip().eq("").sum()),
        "missing_pct": (df.isna().mean() * 100).round(2)
    }))

\ncustomers: shape=(1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,MALE
1,C0002,DELHI,2024-02-17,31,MALE
2,C0003,bengaluru,2025-02-18,18,F


,dtype,missing_count,blank_count,missing_pct
customer_id,str,0,0,0.0
city,str,0,0,0.0
signup_date,str,0,0,0.0
age,str,0,61,0.0
gender,str,0,30,0.0


\nproducts: shape=(40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


,dtype,missing_count,blank_count,missing_pct
product_id,str,0,0,0.0
product_name,str,0,0,0.0
category,str,0,0,0.0
price_inr,int64,0,0,0.0


... (2 outputs left out)

,dtype,missing_count,blank_count,missing_pct
order_id,str,0,0,0.0
customer_id,str,0,0,0.0
product_id,str,0,0,0.0
quantity,int64,0,0,0.0
order_ts,str,0,0,0.0
status,str,0,0,0.0
discount_pct,int64,0,0,0.0
order_amount,str,0,0,0.0
delivery_minutes,str,0,2541,0.0


\ntickets: shape=(164, 5)


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical


,dtype,missing_count,blank_count,missing_pct
ticket_id,str,0,0,0.0
customer_id,str,0,0,0.0
order_id,str,0,8,0.0
text,str,0,4,0.0
true_category,str,0,0,0.0


### Audit questions for students
- Which columns are supposed to be unique keys?
- Are duplicates exact duplicates or multiple states of the same order?
- Which values are impossible (negative delivery duration, non-numeric money, implausible age)?
- Why should we distinguish an empty string from a missing value?
- What would go wrong if we modelled before cleaning?

In [4]:
from src.data_pipeline import audit, clean_data, validate
audit()


customers: shape=(1200, 5)
customer_id    str
city           str
signup_date    str
age            str
gender         str
dtype: object
             missing_count  missing_pct
customer_id              0          0.0
city                     0          0.0
signup_date              0          0.0
age                      0          0.0
gender                   0          0.0

products: shape=(40, 4)
product_id        str
product_name      str
category          str
price_inr       int64
dtype: object
              missing_count  missing_pct
product_id                0          0.0
product_name              0          0.0
category                  0          0.0
price_inr                 0          0.0

orders: shape=(13084, 9)
order_id              str
customer_id           str
product_id            str
quantity            int64
order_ts              str
status                str
discount_pct        int64
order_amount          str
delivery_minutes      str
dtype: object
                 

## 3. Clean the data and write trusted outputs

The pipeline applies explicit rules:
- standardize city and gender;
- make impossible ages missing and fill with city median;
- parse timestamps, remove exact duplicates, then keep the latest row per order ID;
- coerce invalid amounts to missing and calculate replacements from product price × quantity × discount;
- set negative delivery times to missing and impute only completed orders using product-category medians;
- save clean CSVs and SQLite tables.

Ask students to explain why cancelled orders should keep a missing delivery duration.

In [5]:
clean = clean_data()
for name, df in clean.items():
    print(name, df.shape)
    display(df.head(3))
validate()

customers (1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,male
1,C0002,Delhi,2024-02-17,31,male
2,C0003,Bengaluru,2025-02-18,18,female


products (40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


orders (12309, 9)


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
827,ORD-100001,C0001,P001,1,2025-09-16 08:07:21,completed,15,109.65,20
4294,ORD-100002,C0001,P008,1,2025-09-03 07:34:01,completed,40,25.20,23
9973,ORD-100003,C0001,P008,1,2025-04-08 17:08:14,completed,0,42.00,23


ALL CHECKPOINTS PASSED


## 4. Query the SQLite database

The clean tables are persisted in `data/instamart.db`. SQLite gives us a portable analytical database that can be queried without setting up a server.

Run the supplied SQL examples and compare SQL totals to pandas results.

In [9]:
import sqlite3
db_path = ROOT / "data" / "instamart.db"
with sqlite3.connect(db_path) as con:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)
    display(tables)
    q2 = pd.read_sql_query("""
    SELECT 
        c.city, 
        p.category, 
        ROUND(SUM(o.order_amount), 2) AS revenue
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN products p ON o.product_id = p.product_id
    WHERE o.status = 'completed'
    GROUP BY c.city, p.category
    ORDER BY c.city, p.category
""", con)
display(q2)
print("SQL grand total INR:", round(q2.revenue.sum(), 2))

,name
0,customers
1,orders
2,products


,city,category,revenue
0,Bengaluru,Dairy,87075.05
1,Bengaluru,Electronics,233909.45
2,Bengaluru,Grocery,110915.65
3,Bengaluru,Personal Care,20193.85
4,Bengaluru,Snacks,71549.40
5,Chennai,Dairy,78301.80
6,Chennai,Electronics,195508.80
7,Chennai,Grocery,93817.45
8,Chennai,Personal Care,27675.40
9,Chennai,Snacks,68325.70


SQL grand total INR: 2998668.8


## 5. Part B — customer segments and lapse prediction

RFM means:
- **Recency:** days since the most recent completed order at the snapshot date.
- **Frequency:** number of distinct completed orders.
- **Monetary:** total spend across completed orders.

The model module scales RFM for K-means, compares K values, uses PCA to inspect compressed variance, and builds a lapse model. The lapse model's input features use only orders before 2025-09-01; September is reserved to define the label. The scaler and encoder are fitted inside a training pipeline, not on the test data.

In [10]:
from src.customer_models import rfm_and_segments, build_lapse_model
rfm, k_diagnostics, segment_profile, pca_variance = rfm_and_segments(k=4)
display(k_diagnostics.round(3))
display(segment_profile.round(2))
print("PCA explained variance ratio:", pca_variance, "total:", pca_variance.sum())
display(rfm.head())
lapse_summary = build_lapse_model()
lapse_summary

,k,inertia,silhouette
0,2,1820.761,0.465
1,3,1268.235,0.445
2,4,743.071,0.526
3,5,617.367,0.480
4,6,503.171,0.482
5,7,408.460,0.416
6,8,366.503,0.412


,segment_id,size,recency,frequency,monetary
0,0,436,28.86,3.77,1057.62
1,1,138,24.12,6.77,7129.70
2,2,419,8.77,19.40,3410.02
3,3,182,109.04,2.20,685.99


PCA explained variance ratio: [0.63427875 0.21803762] total: 0.8523163667502166


,customer_id,frequency,monetary,recency,segment_id,segment_name,is_premium
0,C0001,4,512.85,14,0,At Risk,False
1,C0002,7,4644.90,28,1,Premium,True
2,C0003,22,3644.85,7,2,Frequent,False
3,C0004,5,369.00,0,0,At Risk,False
4,C0005,2,324.00,147,3,At Risk,False


{'cohort_rows': 1156,
 'lapsed': 443,
 'baseline_accuracy': 0.6163793103448276,
 'models': {'logistic': {'best_params': {'model__C': 100},
   'cv_f1': np.float64(0.650552962900789),
   'precision': 0.6444444444444445,
   'recall': 0.651685393258427,
   'f1': 0.6480446927374302,
   'confusion_matrix': [[111, 32], [31, 58]]},
  'forest': {'best_params': {'model__max_depth': 8,
    'model__min_samples_leaf': 10},
   'cv_f1': np.float64(0.6865517479789224),
   'precision': 0.6333333333333333,
   'recall': 0.6404494382022472,
   'f1': 0.6368715083798883,
   'confusion_matrix': [[110, 33], [32, 57]]}},
 'retention_list':     customer_id  lapse_probability  actually_lapsed
 13        C0862           0.510513                0
 190       C0267           0.477212                0
 165       C1008           0.473260                0
 227       C0060           0.471746                0
 5         C0802           0.471582                0
 216       C0942           0.465245                0
 144   

### Instructor discussion: leakage
Ask students to identify:
1. Which dates can enter the lapse-model features?
2. Why must September orders not enter the features?
3. Why is a Premium-only retention list labelled *illustrative* if the Premium flag comes from whole-window RFM?
4. Why can accuracy be misleading when a substantial minority of customers lapse?

## 6. Part C — support ticket triage

This pipeline demonstrates:
1. input guardrails for empty and overlong tickets;
2. ordered PII redaction (email → phone → order ID);
3. retry logic for simulated transient errors;
4. strict category validation and a keyword fallback;
5. token budgeting and human-review routing.

The exact user message sent to the mock classifier is the redacted ticket text only.

In [11]:
from src.triage import run_pipeline, FEW, ZERO
triage_results, triage_summary = run_pipeline(prompt=FEW, token_budget=100000)
print(triage_summary)
display(triage_results.head(10))
display(triage_results.status.value_counts().rename_axis("status").reset_index(name="tickets"))

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
retry ticket=T0003 retry=0 wait=0.0518s
retry ticket=T0019 retry=0 wait=0.0478s
retry ticket=T0086 retry=0 wait=0.0524s
retry ticket=T0065 retry=0 wait=0.0551s
{'tokens': 12520, 'redactions': {'email': 55, 'phone': 91, 'order_id': 132}, 'status_counts': {'classified_json': 135, 'classified_keyword': 12, 'flagged_invalid_output': 9, 'skipped_empty': 4, 'flagged_long': 4}}


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,At Risk,HIGH,security_team,84
1,T0073,C0080,classified_json,billing,Frequent,NORMAL,support_queue,80
2,T0095,C1087,classified_json,technical,At Risk,NORMAL,support_queue,82
3,T0159,C1172,skipped_empty,,Frequent,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,At Risk,HIGH,security_team,84
5,T0124,C0139,classified_json,fraud,Frequent,HIGH,security_team,84
6,T0061,C0812,classified_json,billing,Frequent,NORMAL,support_queue,76
7,T0118,C0355,classified_json,delivery,Frequent,NORMAL,support_queue,79
8,T0037,C1156,classified_json,billing,Premium,HIGH,premium_desk,80
9,T0002,C0881,classified_json,delivery,At Risk,NORMAL,support_queue,78


,status,tickets
0,classified_json,135
1,classified_keyword,12
2,flagged_invalid_output,9
3,skipped_empty,4
4,flagged_long,4


### Discussion: privacy and safety
- Redaction is a defence-in-depth step, not a guarantee that every sensitive detail is removed.
- Why are invalid categories sent to human review instead of guessed?
- Why should fraud override ordinary customer-segment routing?
- Why do empty/long tickets need to be blocked before any model call?

## 7. Part D — policy RAG

The retrieval layer chunks policy documents, fits a TF-IDF vectorizer and ranks chunks by cosine similarity. It passes retrieved chunks to the offline policy assistant, which should answer only from the context and fall back when a question is not covered.

In [12]:
from src.policy_rag import PolicyRAG, evaluate
rag = PolicyRAG(chunk_size=40, overlap=10)
print("Policy chunks:", len(rag.chunks))
display(pd.DataFrame(rag.chunks).head())
print(rag.answer("What delivery time makes a completed order eligible for a full refund?"))
policy_eval = evaluate(rag)
display(policy_eval)
print("Expected-document hits:", int(policy_eval.retrieved_doc.sum()))
print("Expected-fact hits:", int(policy_eval.fact_found.sum()))
print("Fallbacks:", int(policy_eval.fallback.sum()))

Policy chunks: 15


,source,text
0,delivery_policy.txt,Delivery Policy. Standard delivery is promised...
1,delivery_policy.txt,"of INR 25 is charged on orders below INR 199, ..."
2,delivery_policy.txt,60 minutes and the app shows a rain notice. A ...
3,escalation_policy.txt,Escalation Policy. A ticket about fraud or an ...
4,escalation_policy.txt,"1 hour, and all other tickets within 24 hours...."


{'answer': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount.', 'sources': ['refund_policy.txt', 'privacy_policy.txt'], 'retrieved': [{'source': 'refund_policy.txt', 'score': 0.35464416504570484, 'text': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are'}, {'source': 'refund_policy.txt', 'score': 0.31425507733397284, 'text': 'Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer'}, {'source': 'privacy_policy.txt', 'score': 0.15759666156595936, 'text': 'Delivery addresses are deleted from active systems 180 days after the last order. A customer can ask for account deletion and the request is c

,question,expected_doc,retrieved_doc,fact_found,fallback
0,What delivery time makes a completed order eli...,refund_policy.txt,True,True,False
1,Who must approve a refund above INR 1000?,refund_policy.txt,True,True,False
2,How many days do I have to raise a refund requ...,refund_policy.txt,True,True,False
3,What is the delivery fee on small orders?,delivery_policy.txt,True,True,False
4,On orders of what value is delivery free?,delivery_policy.txt,True,True,False
5,Why does the electronics delivery need a one t...,delivery_policy.txt,True,True,False
6,Is cash on delivery available for a large order?,payment_policy.txt,True,True,False
7,How soon is money returned when a payment fail...,payment_policy.txt,True,True,False
8,Can wallet balance be refunded as cash?,payment_policy.txt,True,True,False
9,When are delivery addresses deleted from activ...,privacy_policy.txt,True,True,False


Expected-document hits: 12
Expected-fact hits: 12
Fallbacks: 2


## 8. Part D — refund tools and safe stopping

The refund agent demonstrates a tool loop: look up an order in SQLite, retrieve policy, calculate the eligible refund, then issue it or escalate to a human. In this teaching kit the ledger is a local JSONL file; no money moves.

Before running agent demonstrations, inspect `src/refund_agent.py` and ask students to trace how a tool name maps to a Python function.

In [13]:
from src.refund_agent import get_order, search_policy, calculate_refund, TOOL_MAP, run_agent
print("Tools:", sorted(TOOL_MAP))
print("Example order lookup:")
with sqlite3.connect(db_path) as con:
    example_order = pd.read_sql_query(
        "SELECT order_id FROM orders WHERE status='completed' AND delivery_minutes > 45 LIMIT 1", con
    ).iloc[0, 0]
print(example_order, get_order(example_order))
print("Refund calculation:", calculate_refund(250.0, 52))
print("Timeout safety demo:", run_agent(f"Refund order {example_order}", scenario_id="timeout-demo", time_limit_seconds=0))

Tools: ['calculate_refund', 'escalate_to_human', 'get_order', 'issue_refund', 'search_policy']
Example order lookup:
ORD-100048 {'order_id': 'ORD-100048', 'status': 'completed', 'order_amount': 128.25, 'delivery_minutes': 65}
Refund calculation: {'refund_inr': 250.0}
Timeout safety demo: {'status': 'FALLBACK_TIMEOUT', 'final_answer': 'FALLBACK_TIMEOUT: hand off to a human agent.', 'steps': [], 'llm_calls': 0, 'tokens_used': 0}


## 9. Final generated artifacts and reproducibility check

Review the CSVs and model file generated by the pipeline. In class, have students delete `data/clean/`, `data/instamart.db`, and `outputs/`, then rerun the documented commands to verify that the project can be rebuilt from source.

**Important:** This teaching kit is an instructional implementation. If the course grades exact numeric checkpoints from the original brief, run every checkpoint and reconcile differences; do not hardcode expected answers.

In [14]:
for folder in [ROOT/"data"/"clean", ROOT/"outputs", ROOT/"state"]:
    print(f"\\n{folder.relative_to(ROOT)}")
    print(sorted(p.name for p in folder.iterdir()) if folder.exists() else "(not created)")
print("Database exists:", db_path.exists())

\ndata\clean
['customers.csv', 'orders.csv', 'products.csv']
\noutputs
['customer_segments.csv', 'lapse_model.joblib', 'retention_list.csv', 'triage_results.csv']
\nstate
[]
Database exists: True


10. Churn Analysis

Customer churn analysis helps a business understand which customers have stopped purchasing and which customers may be at risk of leaving in the future.

In this project, a customer is labelled as **churned** in the descriptive analysis when they have not placed a completed order for at least 90 days by the analysis date. Customers who have purchased more recently are labelled active.

The analysis looks at churn across different customer groups, including city, gender and age group. It also examines how churn relates to support-ticket activity and reviews sales across product categories.

A Random Forest model is then trained to estimate the likelihood that an eligible customer will make no completed purchase during a following 90-day period. It uses historical customer and order information available before the prediction period. Support-ticket information is not included in the predictive model because the sample ticket data does not contain ticket creation dates, so it cannot be reliably limited to information available at prediction time.

The model is evaluated using measures such as accuracy, precision, recall, F1 score and ROC-AUC, along with a confusion matrix and feature-importance results. It also assigns customers to low-, medium- or high-risk groups based on their predicted churn probabilities.

The analysis creates CSV reports containing the overall churn summary, customer-level results and churn comparisons. These outputs can help a business understand customer behaviour and identify areas where retention efforts may be useful. The predictions are estimates, not guarantees that a customer will leave.

In [3]:
# ============================================================
# CUSTOMER CHURN ANALYSIS
# Dataset: customers.csv, orders.csv, products.csv,
#          tickets.csv, policy_eval.csv
# ============================================================

# If necessary, install the required libraries:
# pip install pandas numpy matplotlib seaborn scikit-learn

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

DATA_DIR = "."  # Change this if your CSV files are elsewhere

customers = pd.read_csv(
    os.path.join(DATA_DIR, "customers.csv")
)

orders = pd.read_csv(
    os.path.join(DATA_DIR, "orders.csv")
)

products = pd.read_csv(
    os.path.join(DATA_DIR, "products.csv")
)

tickets = pd.read_csv(
    os.path.join(DATA_DIR, "tickets.csv")
)

policy_eval = pd.read_csv(
    os.path.join(DATA_DIR, "policy_eval.csv")
)

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

for name, df in {
    "Customers": customers,
    "Orders": orders,
    "Products": products,
    "Tickets": tickets,
    "Policy Evaluation": policy_eval
}.items():
    print(f"\n{name}: {df.shape}")
    print(df.head(3))


# ------------------------------------------------------------
# 2. DATA CLEANING
# ------------------------------------------------------------

# Convert date columns
customers["signup_date"] = pd.to_datetime(
    customers["signup_date"], errors="coerce"
)

orders["order_ts"] = pd.to_datetime(
    orders["order_ts"], errors="coerce"
)

# Normalize text columns
for col in ["city", "gender"]:
    customers[col] = customers[col].astype("string").str.strip().str.title()

orders["status"] = (
    orders["status"].astype("string").str.strip().str.lower()
)

# Convert monetary values to numeric
orders["order_amount"] = pd.to_numeric(
    orders["order_amount"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("₹", "", regex=False)
        .str.strip(),
    errors="coerce"
)

orders["quantity"] = pd.to_numeric(
    orders["quantity"], errors="coerce"
)

orders["discount_pct"] = pd.to_numeric(
    orders["discount_pct"], errors="coerce"
)

orders["delivery_minutes"] = pd.to_numeric(
    orders["delivery_minutes"], errors="coerce"
)

# Remove duplicate records
customers = customers.drop_duplicates(
    subset=["customer_id"]
)

orders = orders.drop_duplicates(
    subset=["order_id"]
)

products = products.drop_duplicates(
    subset=["product_id"]
)

tickets = tickets.drop_duplicates(
    subset=["ticket_id"]
)

# Keep orders that can be assigned to a customer and a date
orders = orders.dropna(
    subset=["customer_id", "order_ts"]
)

# Analysis cutoff is the final date in the dataset
AS_OF_DATE = orders["order_ts"].max().normalize()

# Define a 90-day inactivity threshold
CHURN_DAYS = 90

print("\nAnalysis cutoff:", AS_OF_DATE.date())
print("Churn inactivity threshold:", CHURN_DAYS, "days")

print("\nOrder status distribution:")
print(orders["status"].value_counts(dropna=False))


# ------------------------------------------------------------
# 3. DESCRIPTIVE CHURN ANALYSIS
# ------------------------------------------------------------

# Use completed orders to measure purchasing behavior
completed_orders = orders[
    orders["status"] == "completed"
].copy()

# Calculate order-level metrics for each customer
order_features = completed_orders.groupby(
    "customer_id"
).agg(
    total_completed_orders=("order_id", "nunique"),
    total_spend=("order_amount", lambda x: x.sum(min_count=1)),
    average_order_value=("order_amount", "mean"),
    total_quantity=("quantity", "sum"),
    average_discount=("discount_pct", "mean"),
    average_delivery_minutes=("delivery_minutes", "mean"),
    first_order_date=("order_ts", "min"),
    last_order_date=("order_ts", "max")
).reset_index()

# Count all orders by status
status_counts = orders.pivot_table(
    index="customer_id",
    columns="status",
    values="order_id",
    aggfunc="nunique",
    fill_value=0
)

status_counts.columns = [
    f"{str(col)}_orders" for col in status_counts.columns
]

status_counts = status_counts.reset_index()

# Combine customer information and order metrics
customer_analysis = customers.merge(
    order_features,
    on="customer_id",
    how="left"
)

customer_analysis = customer_analysis.merge(
    status_counts,
    on="customer_id",
    how="left"
)

# Fill missing order counts with zero
count_columns = [
    col for col in customer_analysis.columns
    if col.endswith("_orders") or col == "total_quantity"
]

customer_analysis[count_columns] = (
    customer_analysis[count_columns].fillna(0)
)

# Calculate days since the last completed purchase
customer_analysis["days_since_last_order"] = (
    AS_OF_DATE - customer_analysis["last_order_date"]
).dt.days

# Customers without completed purchases are treated as inactive
customer_analysis["days_since_last_order"] = (
    customer_analysis["days_since_last_order"].fillna(
        (AS_OF_DATE - customer_analysis["signup_date"]).dt.days
    )
)

# Calculate customer tenure
customer_analysis["customer_tenure_days"] = (
    AS_OF_DATE - customer_analysis["signup_date"]
).dt.days

# Avoid classifying customers who have not yet reached
# the inactivity threshold as churned.
customer_analysis["churn"] = (
    customer_analysis["days_since_last_order"] >= CHURN_DAYS
).astype(int)

customer_analysis["churn_status"] = (
    customer_analysis["churn"]
    .map({1: "Churned", 0: "Active"})
)

# ------------------------------------------------------------
# 4. OVERALL CHURN METRICS
# ------------------------------------------------------------

total_customers = len(customer_analysis)

churned_customers = int(
    customer_analysis["churn"].sum()
)

active_customers = total_customers - churned_customers

churn_rate = (
    churned_customers / total_customers * 100
    if total_customers > 0 else 0
)

print("\n" + "=" * 60)
print("OVERALL CHURN SUMMARY")
print("=" * 60)

print("Total customers:", total_customers)
print("Churned customers:", churned_customers)
print("Active customers:", active_customers)
print(f"Churn rate: {churn_rate:.2f}%")
print(f"Retention rate: {100 - churn_rate:.2f}%")

print("\nChurn status distribution:")
print(customer_analysis["churn_status"].value_counts())


# ------------------------------------------------------------
# 5. CHURN BY CITY
# ------------------------------------------------------------

city_churn = customer_analysis.groupby(
    "city", dropna=False
).agg(
    total_customers=("customer_id", "nunique"),
    churned_customers=("churn", "sum"),
    average_spend=("total_spend", "mean")
).reset_index()

city_churn["churn_rate_pct"] = (
    city_churn["churned_customers"]
    / city_churn["total_customers"].replace(0, np.nan)
    * 100
)

city_churn = city_churn.sort_values(
    "churn_rate_pct", ascending=False
)

print("\n" + "=" * 60)
print("CHURN BY CITY")
print("=" * 60)
print(city_churn.to_string(index=False))


# ------------------------------------------------------------
# 6. CHURN BY GENDER
# ------------------------------------------------------------

gender_churn = customer_analysis.groupby(
    "gender", dropna=False
).agg(
    total_customers=("customer_id", "nunique"),
    churned_customers=("churn", "sum")
).reset_index()

gender_churn["churn_rate_pct"] = (
    gender_churn["churned_customers"]
    / gender_churn["total_customers"].replace(0, np.nan)
    * 100
)

print("\n" + "=" * 60)
print("CHURN BY GENDER")
print("=" * 60)
print(gender_churn.to_string(index=False))


# ------------------------------------------------------------
# 7. CHURN BY AGE GROUP
# ------------------------------------------------------------

customer_analysis["age_group"] = pd.cut(
    customer_analysis["age"],
    bins=[0, 24, 34, 44, 54, 120],
    labels=[
        "18-24", "25-34", "35-44", "45-54", "55+"
    ],
    include_lowest=True
)

age_churn = customer_analysis.groupby(
    "age_group", observed=False, dropna=False
).agg(
    total_customers=("customer_id", "nunique"),
    churned_customers=("churn", "sum")
).reset_index()

age_churn["churn_rate_pct"] = (
    age_churn["churned_customers"]
    / age_churn["total_customers"].replace(0, np.nan)
    * 100
)

print("\n" + "=" * 60)
print("CHURN BY AGE GROUP")
print("=" * 60)
print(age_churn.to_string(index=False))


# ------------------------------------------------------------
# 8. SUPPORT TICKET ANALYSIS
# ------------------------------------------------------------

# Count tickets by customer
ticket_features = tickets.groupby(
    "customer_id"
).agg(
    total_tickets=("ticket_id", "nunique"),
    ticket_categories=("true_category", "nunique")
).reset_index()

customer_analysis = customer_analysis.merge(
    ticket_features,
    on="customer_id",
    how="left"
)

customer_analysis["total_tickets"] = (
    customer_analysis["total_tickets"].fillna(0)
)

customer_analysis["ticket_categories"] = (
    customer_analysis["ticket_categories"].fillna(0)
)

# Note: this is a descriptive association.
# Tickets may have been created after a customer's inactivity
# started, so ticket counts are not used in the predictive model.

ticket_churn = customer_analysis.groupby(
    customer_analysis["total_tickets"] > 0
).agg(
    total_customers=("customer_id", "nunique"),
    churned_customers=("churn", "sum"),
    average_spend=("total_spend", "mean")
).reset_index()

ticket_churn["ticket_group"] = ticket_churn[
    "total_tickets"
].map({
    True: "Has support tickets",
    False: "No support tickets"
})

ticket_churn["churn_rate_pct"] = (
    ticket_churn["churned_customers"]
    / ticket_churn["total_customers"].replace(0, np.nan)
    * 100
)

print("\n" + "=" * 60)
print("CHURN AND SUPPORT TICKETS")
print("=" * 60)
print(ticket_churn.to_string(index=False))


# ------------------------------------------------------------
# 9. PRODUCT CATEGORY ANALYSIS
# ------------------------------------------------------------

product_orders = completed_orders.merge(
    products[["product_id", "product_name", "category"]],
    on="product_id",
    how="left"
)

category_analysis = product_orders.groupby(
    "category", dropna=False
).agg(
    total_orders=("order_id", "nunique"),
    total_quantity=("quantity", "sum"),
    total_revenue=("order_amount", "sum")
).reset_index()

category_analysis = category_analysis.sort_values(
    "total_revenue", ascending=False
)

print("\n" + "=" * 60)
print("PRODUCT CATEGORY PERFORMANCE")
print("=" * 60)
print(category_analysis.to_string(index=False))


# ------------------------------------------------------------
# 10. VISUALIZE CHURN
# ------------------------------------------------------------

# Chart 1: Churn distribution
plt.figure(figsize=(7, 5))

sns.countplot(
    data=customer_analysis,
    x="churn_status"
)

plt.title("Customer Churn Distribution")
plt.xlabel("Customer Status")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()


# Chart 2: Churn by city
plot_city = city_churn.dropna(
    subset=["city", "churn_rate_pct"]
)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=plot_city,
    x="city",
    y="churn_rate_pct"
)

plt.title("Customer Churn Rate by City")
plt.xlabel("City")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# Chart 3: Churn by age group
plot_age = age_churn.dropna(
    subset=["age_group", "churn_rate_pct"]
)

plt.figure(figsize=(8, 5))

sns.barplot(
    data=plot_age,
    x="age_group",
    y="churn_rate_pct"
)

plt.title("Customer Churn Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()


# Chart 4: Revenue by product category
plt.figure(figsize=(10, 6))

sns.barplot(
    data=category_analysis,
    x="category",
    y="total_revenue"
)

plt.title("Revenue by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 11. PREDICTIVE CHURN MODEL
# ------------------------------------------------------------
# A more defensible setup:
#
# Observation window: all activity up to a snapshot date.
# Prediction window: the following 90 days.
#
# Label = 1 if a customer makes no completed purchase
# during the following 90 days.
#
# Only customers registered on or before the snapshot date
# are included. All model features use information available
# on or before that date.

PREDICTION_DAYS = 90

SNAPSHOT_DATE = (
    AS_OF_DATE - pd.Timedelta(days=PREDICTION_DAYS)
)

PREDICTION_END = SNAPSHOT_DATE + pd.Timedelta(
    days=PREDICTION_DAYS
)

print("\n" + "=" * 60)
print("PREDICTIVE MODEL SETUP")
print("=" * 60)

print("Observation cutoff:", SNAPSHOT_DATE.date())
print("Prediction window ends:", PREDICTION_END.date())

# Eligible customers must have registered by the snapshot date
eligible_customers = customers[
    customers["signup_date"].notna()
    & (customers["signup_date"] <= SNAPSHOT_DATE)
].copy()

# Orders available at prediction time
past_orders = orders[
    (orders["order_ts"] <= SNAPSHOT_DATE)
].copy()

# Completed orders used to build historical purchase features
past_completed = past_orders[
    past_orders["status"] == "completed"
].copy()

# Build features from historical completed orders
past_features = past_completed.groupby(
    "customer_id"
).agg(
    historical_completed_orders=("order_id", "nunique"),
    historical_spend=("order_amount", lambda x: x.sum(min_count=1)),
    historical_avg_order_value=("order_amount", "mean"),
    historical_avg_discount=("discount_pct", "mean"),
    historical_avg_delivery=("delivery_minutes", "mean"),
    historical_last_order=("order_ts", "max")
).reset_index()

# Count historical order statuses
past_status_counts = past_orders.pivot_table(
    index="customer_id",
    columns="status",
    values="order_id",
    aggfunc="nunique",
    fill_value=0
)

past_status_counts.columns = [
    f"past_{col}_orders" for col in past_status_counts.columns
]

past_status_counts = past_status_counts.reset_index()

# Only use tickets created on or before the snapshot.
# The dataset has no ticket creation timestamp, so we cannot
# safely include ticket information in the prediction model.

model_data = eligible_customers.merge(
    past_features,
    on="customer_id",
    how="left"
)

model_data = model_data.merge(
    past_status_counts,
    on="customer_id",
    how="left"
)

# Historical ticket data is intentionally excluded from
# the model because ticket timestamps are not available.

# Calculate recency at the snapshot date
model_data["historical_recency_days"] = (
    SNAPSHOT_DATE - model_data["historical_last_order"]
).dt.days

model_data["historical_recency_days"] = (
    model_data["historical_recency_days"].fillna(
        (SNAPSHOT_DATE - model_data["signup_date"]).dt.days
    )
)

model_data["tenure_days"] = (
    SNAPSHOT_DATE - model_data["signup_date"]
).dt.days

# Replace missing historical order counts with zero
model_count_cols = [
    col for col in model_data.columns
    if col.startswith("past_")
    and col.endswith("_orders")
]

model_data[model_count_cols] = (
    model_data[model_count_cols].fillna(0)
)

model_data["historical_completed_orders"] = (
    model_data["historical_completed_orders"].fillna(0)
)

# Create labels using the next 90 days
future_orders = orders[
    (orders["order_ts"] > SNAPSHOT_DATE)
    & (orders["order_ts"] <= PREDICTION_END)
    & (orders["status"] == "completed")
]

future_purchase_counts = future_orders.groupby(
    "customer_id"
)["order_id"].nunique()

model_data["future_completed_orders"] = (
    model_data["customer_id"]
    .map(future_purchase_counts)
    .fillna(0)
)

model_data["churn_target"] = (
    model_data["future_completed_orders"] == 0
).astype(int)

print("Eligible customers:", len(model_data))
print(
    "Future-window churn rate:",
    f"{model_data['churn_target'].mean() * 100:.2f}%"
    if len(model_data) else "Not available"
)

# Exclude identifiers, dates, and future information.
# Recency is included because it is known at the snapshot.
# It does not directly define the future target.
#
# Historical purchase features are available before prediction.
# No future order counts are used as model predictors.

categorical_features = [
    "city",
    "gender"
]

numeric_features = [
    "age",
    "tenure_days",
    "historical_recency_days",
    "historical_completed_orders",
    "historical_spend",
    "historical_avg_order_value",
    "historical_avg_discount",
    "historical_avg_delivery"
] + model_count_cols

feature_columns = categorical_features + numeric_features

# Ensure columns exist
for col in feature_columns:
    if col not in model_data.columns:
        model_data[col] = np.nan

X = model_data[feature_columns].copy()
y = model_data["churn_target"].copy()

# Check that enough data and both target classes exist
if (
    len(model_data) < 30
    or y.nunique() < 2
    or y.value_counts().min() < 2
):
    print(
        "\nNot enough examples from both classes to train "
        "a reliable churn model. Use the descriptive analysis "
        "above, or collect more historical data."
    )

else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y
    )

    # Numeric preprocessing
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]
    )

    # Categorical preprocessing
    categorical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "encoder",
                OneHotEncoder(handle_unknown="ignore")
            )
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_transformer,
                numeric_features
            ),
            (
                "cat",
                categorical_transformer,
                categorical_features
            )
        ]
    )

    # Random Forest classifier
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=300,
                    class_weight="balanced",
                    random_state=42,
                    min_samples_leaf=2
                )
            )
        ]
    )

    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    y_probability = model.predict_proba(X_test)[:, 1]

    print("\n" + "=" * 60)
    print("CHURN MODEL PERFORMANCE")
    print("=" * 60)

    print(
        "Accuracy:",
        round(accuracy_score(y_test, y_pred), 4)
    )

    print(
        "Precision:",
        round(
            precision_score(
                y_test, y_pred, zero_division=0
            ),
            4
        )
    )

    print(
        "Recall:",
        round(
            recall_score(
                y_test, y_pred, zero_division=0
            ),
            4
        )
    )

    print(
        "F1 Score:",
        round(
            f1_score(
                y_test, y_pred, zero_division=0
            ),
            4
        )
    )

    if y_test.nunique() == 2:
        print(
            "ROC-AUC:",
            round(
                roc_auc_score(y_test, y_probability),
                4
            )
        )

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=["Retained", "Churned"],
            zero_division=0
        )
    )

    # Confusion matrix
    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    )

    plt.figure(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=["Retained", "Churned"],
        yticklabels=["Retained", "Churned"]
    )

    plt.title("Churn Model Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

    # Feature importance
    fitted_preprocessor = model.named_steps["preprocessor"]

    feature_names = fitted_preprocessor.get_feature_names_out()

    importances = (
        model.named_steps["classifier"].feature_importances_
    )

    feature_importance = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values(
        "importance", ascending=False
    )

    print("\nTop 15 Important Features:")
    print(feature_importance.head(15).to_string(index=False))

    plt.figure(figsize=(10, 7))

    sns.barplot(
        data=feature_importance.head(15),
        x="importance",
        y="feature"
    )

    plt.title("Top 15 Churn Predictors")
    plt.xlabel("Feature Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

    # Risk scores for all eligible customers
    model_data["predicted_churn_probability"] = (
        model.predict_proba(X)[:, 1]
    )

    model_data["risk_category"] = pd.cut(
        model_data["predicted_churn_probability"],
        bins=[-0.001, 0.30, 0.60, 1.001],
        labels=["Low", "Medium", "High"]
    )

    print("\nChurn Risk Distribution:")
    print(
        model_data["risk_category"]
        .value_counts()
        .sort_index()
    )

    # Export predictive model results
    model_data.to_csv(
        "churn_prediction_results.csv",
        index=False
    )

    feature_importance.to_csv(
        "churn_feature_importance.csv",
        index=False
    )

    print("\nSaved:")
    print("- churn_prediction_results.csv")
    print("- churn_feature_importance.csv")


# ------------------------------------------------------------
# 12. EXPORT DESCRIPTIVE ANALYSIS RESULTS
# ------------------------------------------------------------

customer_analysis.to_csv(
    "customer_churn_analysis.csv",
    index=False
)

city_churn.to_csv(
    "churn_by_city.csv",
    index=False
)

gender_churn.to_csv(
    "churn_by_gender.csv",
    index=False
)

age_churn.to_csv(
    "churn_by_age_group.csv",
    index=False
)

category_analysis.to_csv(
    "product_category_analysis.csv",
    index=False
)

ticket_churn.to_csv(
    "churn_by_support_tickets.csv",
    index=False
)

# Summary report
summary = pd.DataFrame({
    "metric": [
        "Analysis cutoff",
        "Churn threshold (days)",
        "Total customers",
        "Churned customers",
        "Active customers",
        "Churn rate (%)",
        "Retention rate (%)"
    ],
    "value": [
        str(AS_OF_DATE.date()),
        CHURN_DAYS,
        total_customers,
        churned_customers,
        active_customers,
        round(churn_rate, 2),
        round(100 - churn_rate, 2)
    ]
})

summary.to_csv(
    "churn_summary.csv",
    index=False
)

print("\n" + "=" * 60)
print("CHURN ANALYSIS COMPLETED")
print("=" * 60)

print("Exported descriptive analysis files:")
print("- customer_churn_analysis.csv")
print("- churn_by_city.csv")
print("- churn_by_gender.csv")
print("- churn_by_age_group.csv")
print("- product_category_analysis.csv")
print("- churn_by_support_tickets.csv")
print("- churn_summary.csv")

FileNotFoundError: [Errno 2] No such file or directory: '.\\customers.csv'